# Ingest silver stream

In [1]:
import sys
sys.path.append('/home/nblong/Dropbox/Project/ops/ml-ops/src')

import os
from datetime import datetime, timezone

from delta.tables import DeltaTable
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from bronze.ingest_bronze import SCHEMA as BRONZE_SCHEMA
from bronze.ingest_bronze import TABLE_BATCH as BRONZE_TABLE_BATCH
from bronze.ingest_bronze import TABLE_STREAM as BRONZE_TABLE_STREAM
from silver.silver_map import (
    payment_map,
    rate_code_id_map,
    store_fwd_flag_map,
    vendor_id_map,
)
from silver.valid_silver import post_validate_silver
from utils.utils import _get_table_version, check_date, parse_args


In [2]:
SCHEMA_STORE = 'warehouse'
CHECKPOINT = '/home/nblong/Dropbox/Project/ops/ml-ops/spark/checkpoint/silver_stream'

spark = (
    SparkSession.builder
    .appName('ingest data to bronze zone')
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")

    # packages supporting read/write postgres, S3, hive (data lake)
    # and Kafka, avro (for streaming)
    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.4.2,'
        'software.amazon.awssdk:bundle:2.29.52,'
        'io.delta:delta-spark_4.1_2.13:4.3.0,'
    )

    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))

    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    # allow rewrite partitions
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")

    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")

    # workaround for Spark 4.1.x CollapseProject optimizer bug
    .config("spark.sql.optimizer.excludedRules", "org.apache.spark.sql.catalyst.optimizer.CollapseProject")

    # stream support
    .config('spark.sql.streaming.checkpointLocation', CHECKPOINT)
    .config('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true')

    # create schema using spark (instead of using trino)
    .config("spark.sql.warehouse.dir", os.path.join(os.getenv('S3_URI'), SCHEMA_STORE))
    .enableHiveSupport()
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/29 16:24:21 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.36.108 instead (on interface wlp0s20f3)
26/09/29 16:24:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nblong/.ivy2.5.2/cache
The jars for the packages stored in: /home/nblong/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f8c16503-8a14-4e38-89cd-bc0235fc13fb;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundl

In [3]:
SCHEMA = "silver"

# batch
TABLE_BATCH  = "nyc_taxi"
SCHEMA_STORE = 'warehouse'
S3_SILVER_BATCH_PATH = os.path.join(os.getenv('S3_URI'), SCHEMA)

# stream
TABLE_STREAM = "nyc_taxi_stream"
S3_SILVER_STREAM_PATH = os.path.join(os.getenv('S3_URI'), f'{SCHEMA}_stream')

In [4]:
def _transform(df:DataFrame)->DataFrame:
    # cast some cols to their natural datatype
    cast_map = {
        'passenger_count': F.col('passenger_count').cast(T.IntegerType()),
        'pu_location_id' : F.col('pu_location_id').cast(T.IntegerType()),
        'do_location_id' : F.col('do_location_id').cast(T.IntegerType()),
    }
    df = df.withColumns(cast_map)

    # payment_type
    df = (
        df
        .withColumn('payment_type', F.lower(F.trim(F.col('payment_type'))))
        .replace(payment_map, subset='payment_type')
    )

    # drop outlier for `passenger_count`
    df = df.filter(
        (F.col('passenger_count')<7) &
        (F.col('passenger_count')!=0)
    )

    # drop outlier for `rate_code_id` and cast to int
    df = (
        df.filter(
            ((F.col('rate_code_id')>=1) & (F.col('rate_code_id')<=6)) | (F.col('rate_code_id')==99) |
            (F.col('rate_code_id').isNull())
        )
        .withColumn('rate_code_id', F.col('rate_code_id').cast('string'))
        .replace(rate_code_id_map, subset='rate_code_id')
        .withColumnRenamed('rate_code_id', 'rate_code')
    )

    # store_and_fwd_flag
    df = (
        df
        .withColumn('store_and_fwd_flag', F.lower(F.trim(F.col('store_and_fwd_flag'))))
        .replace(store_fwd_flag_map, subset='store_and_fwd_flag')
    )

    # assign int id for vendor (instead of using string)
    df = (
        df
        .withColumnRenamed('vendor_id', 'vendor')
        .withColumn('vendor', F.lower(F.trim(F.col('vendor'))))
        .replace(vendor_id_map, subset='vendor')
    )

    # add audit fields of silver zone
    batch_timestamp = datetime.now(timezone.utc)
    df = (
        df.withColumn('silver_created_datetime', F.lit(batch_timestamp).cast('timestamp'))
        .withColumn('silver_updated_datetime', F.lit(batch_timestamp).cast('timestamp'))
    )

    # create primary key (sorted columns for deterministic hash)
    # đáng nhé nên thêm `silver_created_datetime` vào `audit_cols` nhưng tôi muốn tạo ra các records bị update giả
    # nếu thêm `silver_created_datetime` thì data sẽ không được update nếu chạy lại (vì đây là dữ liệu tĩnh nên tôi muốn chạy lại để thử logic upsert)
    audit_cols = {'year', 'month', 'source_file', 'source_created_datetime', 'bronze_created_datetime', 'silver_updated_datetime'}
    df = df.withColumn(
        'content_hash',
        F.sha2(F.concat_ws(
            "||",
            *sorted(set(df.columns).difference(audit_cols))
        ), 256)
    )

    # de-dup
    return (
        df
        # .drop('bronze_created_datetime')
        .dropDuplicates(['content_hash'])
    )

def _insert_batch(df:DataFrame, target_table:str, s3_path:str)->int:
    (
        df
        .repartition("year", "month") # prevent small files problem
        .write
        .option("delta.enableDeletionVectors", "true") # have deletion vectors on
        .format('delta')
        .partitionBy("year", "month")
        .mode("append")
        .option("path", s3_path)
        .saveAsTable(target_table)   # đăng ký luôn với Hive metastore
    )
    return df.count()

def _upsert_batch(spark:SparkSession, df:DataFrame, target_table:str):
    delta_table = DeltaTable.forName(spark, target_table)

    # upsert to delta table
    # những row thoả điều kiện sẽ được update các trường như định nghĩa
    # những row không thoả điều kiện sẽ được insert
    partitions = df.select("year", "month").distinct().collect()
    partition_filter = " or ".join([
        f"(target.year={r['year']} and target.month={r['month']})"
        for r in partitions
    ])
    target_before_count = delta_table.toDF().count()
    (
        delta_table.alias("target")
        .merge(
            source=df.alias("source"),
            condition=f"({partition_filter}) and (target.record_key = source.record_key)"
        )
        .whenMatchedUpdate(
            condition='target.content_hash != source.content_hash',
            set={
                col: f"source.{col}"
                for col in df.columns
                if col not in ['month', 'year', 'bronze_created_datetime', 'record_key', 'source_file', 'source_created_datetime', "silver_created_datetime"]
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    target_after_count = delta_table.toDF().count()
    source_count = df.count()
    return target_before_count, target_after_count, source_count

def _save_batch(spark:SparkSession, df:DataFrame, target_table:str, s3_path:str)->tuple[int, int]:
    previous_version = _get_table_version(spark, target_table)
    if previous_version == -1:
        source_count = _insert_batch(df, target_table, s3_path)
        print(f"Table registered as '{target_table}' → {s3_path}\n")
        print(f"\tInserted {source_count} rows")
    else:
        target_before_count, target_after_count, source_count = _upsert_batch(spark, df, target_table)
        print(f"Table registered as '{target_table}' → {S3_SILVER_BATCH_PATH}")
        print(f"\tTarget before operation: {target_before_count} rows. Target after operation: {target_after_count} rows")
        print(f"\tUpdated {target_before_count + source_count - target_after_count} rows")
        print(f"\tInserted {target_after_count - target_before_count} rows")
    current_version = _get_table_version(spark, target_table)
    return source_count, current_version if current_version != previous_version else -1

In [ ]:
def ingest_stream_silver(
    spark:SparkSession,
    source_table:str,
    target_table:str,
    target_s3_path:str,
    checkpoint:str
):
    bronze_df = (
        spark.readStream
        .format('delta')
        .option("startingVersion", "latest") # read from the latest version of table
        .table(source_table)
    )

    silver_df = _transform(bronze_df)

    def _save_stream(batch_df:DataFrame, batch_id:int):
        if batch_df.count() != 0:
            source_count, current_version = _save_batch(spark, batch_df, target_table, target_s3_path)
            if source_count != 0 and current_version != -1:
                post_validate_silver(spark, target_table, source_count, current_version)

    query = (
        silver_df.writeStream
        .foreachBatch(_save_stream)
        .trigger(processingTime='30 seconds')
        .option("checkpointLocation", checkpoint)
        .start()
    )
    print(f"Table registered as '{target_table}' → {target_s3_path}")
    query.awaitTermination()

In [6]:
source_table = f'{BRONZE_SCHEMA}.{BRONZE_TABLE_STREAM}'
target_table = f'{SCHEMA}.{TABLE_STREAM}'
ingest_stream_silver(
    spark=spark,
    source_table=source_table,
    target_table=target_table,
    target_s3_path=S3_SILVER_STREAM_PATH,
    checkpoint=CHECKPOINT
)

26/09/29 16:24:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Table registered as 'silver.nyc_taxi_stream' → s3a://gestore-stuff/nyc_taxi/silver_stream


26/09/29 16:24:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/29 16:24:52 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/09/29 16:25:18 ERROR MicroBatchExecution: Query [id = c0673b52-251d-4667-b416-694e66d26c46, runId = c86e9cb1-1a58-4e24-9847-30054d5deb0e] terminated with error
org.apache.spark.sql.execution.streaming.sources.ForeachBatchUserFuncException: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the user provided function in foreach batch sink. Reason: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

StreamingQueryException: [STREAM_FAILED] Query [id = c0673b52-251d-4667-b416-694e66d26c46, runId = c86e9cb1-1a58-4e24-9847-30054d5deb0e] terminated with exception: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the user provided function in foreach batch sink. Reason: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_995853/2348238655.py", l